In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

import matplotlib.pyplot as plt
%matplotlib inline

from colibri_n3fit.api import API

# API examples for NTK utilities
--------------------------------

## Compute the ensemble of eigenvectors of the NTK at a given epoch

The provider function `eigenvectors_ensemble_at_epoch` allows to compute the ensemble of eigenvectors of the NTK at a given epoch for a specified Monte Carlo fit. The minimal required arguments are the fit identifier and the epoch number (see below). It returns a dictionary with the following items:
- `eigenvectors_data`: ndarray (n_replicas, n_eigenvectors, n_flav * n_xgrid)
- `epoch`: the epoch number
- `ntk_shape`: shape of the NTK matrix before flattening
- `replica_indices`: list of replica indices included

In [ ]:
result_dict = API.eigenvectors_ensemble_at_epoch(fit="260123-ac-nnpdf40-dis-ntk", epoch=100, max_workers=5)
print(f"Shape of raw data {result_dict['eigenvectors_data'].shape}")
print(f"Epoch: {result_dict['epoch']}")
print(f"Shape of the NTK {result_dict['shape']}")
print(f"Number of replicas: {len(result_dict['replica_indices'])}")

In addition, the user can also specify:
- `replica_index_list`: tuple of replica indices to include in the computation. If not provided, all replicas available for the fit will be used.
- `max_workers`: number of parallel workers to use for the computation. Default is `min(10, n_replicas).

Note that the `replica_index_list` is mainly meant for testing purposes with a reduced number of replicas. The example below shows how to compute the eigenvectors for a subset of the replica ensemble at epoch 100 using 5 parallel workers:

In [ ]:
result_dict = API.eigenvectors_ensemble_at_epoch(fit="260123-ac-nnpdf40-dis-ntk", epoch=100, replica_index_list=(10,1,75,3,23), max_workers=5)
print(f"Shape of raw data {result_dict['eigenvectors_data'].shape}")
print(f"Epoch: {result_dict['epoch']}")
print(f"Shape of the NTK {result_dict['shape']}")
print(f"Number of replicas: {len(result_dict['replica_indices'])}")

### Plot the eigenvectors by fit and flavour or by rank and flavour

The eigenvectors of the NTK can be visualised using two different plotting functions: `plot_eigenvectors_by_fit_and_flavour` and `plot_eigenvectors_by_rank_and_flavour`. The first function
allows to compare a fixed set of eigenvector ranks across different fits for specified flavours. The second function allows to compare different eigenvector ranks within the same fit for specified flavours. Both functions require the following arguments:
- `fits`: a list of dictionaries, each containing the fit identifier and an optional label for the legend.
- `rank_indices`: a list of eigenvector ranks to plot.
- `flavour_mapping`: a list of flavour codes to include in the plots.
- `epoch`: the epoch number at which to compute the eigenvectors.

In addition, on top of the additional arguments `max_workers` and `replica_index_list` described above, the user can also specify:
- `error_type`: type of error to display. Available options are:
  - `mean`, that shows the mean and standard deviation across replicas.
  - `median`, that shows the median and 68% confidence interval across replicas. Default is `mean`.
- `xscale`, `yscale`: scale of the x- and y-axis. Default is `linear.
- `ymin`, `ymax`: minimum and maximum values for the y-axis. Default is `None`.

In [ ]:
res = API.plot_eigenvectors_by_rank_and_flavour(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], 
                                                rank_indices=[0, 1, 2], 
                                                flavour_mapping=['g', 'T3'], 
                                                epoch=100,
                                                error_type='mean',)
results = {r[1].name : (r[0], r[1]) for r in res}

In [ ]:
results['eigvec_1_g'][1].ax.set_title("Eigenvector 1, gluon")
results['eigvec_1_g'][0]

In [ ]:
results['eigvec_2_T3'][1].ax.set_title("Eigenvector 1, T3")
results['eigvec_2_T3'][0]

In [ ]:
gen_eigvec_by_fit = API.plot_eigenvectors_by_fit_and_flavour(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], 
                                                rank_indices=[0, 1, 2], 
                                                flavour_mapping=['g', 'T3'], 
                                                epoch=100,
                                                error_type='mean',)
results_eigvec_by_fit = {r[1].name : (r[0], r[1]) for r in gen_eigvec_by_fit}

In [ ]:
results_eigvec_by_fit['eigvecs_T3'][0]

# Compute the eigenvalues of the NTK

Similarly to the eigenvectors, the eigenvalue of the NTK can be computed and visualised accordingly. For instance, the user can use the provider function `eigenvalues_ensemble` to compute the ensemble of eigenvalues of the NTK for a specified Monte Carlo fit across **all** specified epochs. The function can be called with the following arguments:
- `fit`: fit identifier.
- `replica_index_list`: tuple of replica indices to include in the computation. If not provided, all replicas available for the fit will be used. It is mainly meant for testing purposes with a reduced number of replicas.
- `max_epoch`: maximum epoch to consider. Replicas that do not reach this epoch will be excluded from the computation. Default is `None`, meaning that the epochs considered will be those in the intersection of all replicas.
- `max_workers`: number of parallel workers to use for the computation. Default is `min(10, n_replicas).
- `force_recompute`: whether to force recomputation of the eigenvalues even if cached data is available. Default is `False`.

Note that, contrary to the case of the eigenvectors, the eigenvalue are stored in the folders of each replicas. If the users wants to recompute them using, for instance, a different common epochs rule, the `force_recompute` flag should be set to `True`.

The function returns a dictionary with the following items:
- `eigenvalues_by_epoch`: dict mapping epoch -> ndarray (n_replicas, n_eigenvalues)
- `epochs`: list of epochs
- `ntk_shape`: shape of NTK matrix
- `replica_indices`: list of replica indices included

In [ ]:
eigvals_dict = API.eigenvalues_ensemble(fit="260123-ac-nnpdf40-dis-ntk", force_recompute=False)
print(f"Shape of raw data {eigvals_dict['eigenvalues_by_epoch'][0].shape}")
print(f"Number of epochs: {len(eigvals_dict['epochs'])} <----")
print(f"Maximum epoch: {max(eigvals_dict['epochs'])}")
print(f"Shape of the NTK {eigvals_dict['ntk_shape']}")
print(f"Number of replicas: {len(eigvals_dict['replica_indices'])}")

Note that in this case, we have not specified `max_epoch`, so all epochs in the intersection of all replicas are considered. This results in just 21 epochs considered, with a maximum epoch of 1000.

The eigenvalues can then be visualised using the plotting function `plot_eigvals_by_fit` or `plot_eigvals_by_rank`, which work similarly to the eigenvector plotting functions described above.

In [ ]:
gen_eigval = API.plot_eigvals_by_rank(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], rank_indices=[0, 1, 2])
results_eigval = {r[1].name : (r[0], r[1]) for r in gen_eigval}

In [ ]:
results_eigval['lambda_1'][0]

In [ ]:
gen_eigval_by_fit = API.plot_eigvals_by_fit(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], rank_indices=[0, 1, 2], error_type='median', yscale='log')
results_eigval_by_fit = {r[1].name : (r[0], r[1]) for r in gen_eigval_by_fit}

In [ ]:
results_eigval_by_fit['eigvals_$\\textrm{Custom label for fit}$'][0]

In addition, there is an extra utility function, `plot_eigvals_replicas_by_rank`, that shows each replica's eigenvalues for a given fit and rank index across epochs.

In [ ]:
generator_eigval_replicas = API.plot_eigvals_replicas_by_rank(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], rank_indices=[0])
results_eigval_replicas = {r[1].name : (r[0], r[1]) for r in generator_eigval_replicas}

In [ ]:
results_eigval_replicas['lambda_replicas_1'][1].ax.set_yscale('linear')
results_eigval_replicas['lambda_replicas_1'][0]

Finally, we can request a higher maximum epoch. This will filter out replicas that do not reach that epoch.

In [ ]:
eigvals_dict = API.eigenvalues_ensemble(fit="260123-ac-nnpdf40-dis-ntk", force_recompute=False, max_epoch=25000)
print(f"Shape of raw data {eigvals_dict['eigenvalues_by_epoch'][0].shape}")
print(f"Number of epochs: {len(eigvals_dict['epochs'])} <----")
print(f"Maximum epoch: {max(eigvals_dict['epochs'])}")
print(f"Shape of the NTK {eigvals_dict['ntk_shape']}")
print(f"Number of replicas: {len(eigvals_dict['replica_indices'])}")

Note that now we have 501 epochs considered, with a maximum epoch of 25000. This however reduces the number of replicas included in the computation, as some replicas do not reach that epoch.

In [ ]:
gen_eigval = API.plot_eigvals_by_rank(fits=[{"id": "260123-ac-nnpdf40-dis-ntk", "label": r"$\textrm{Custom label for fit}$"}], rank_indices=[0, 1, 2], max_epoch=25000)
results_eigval = {r[1].name : (r[0], r[1]) for r in gen_eigval}

In [ ]:
results_eigval['lambda_1'][0]